# Day 1: Advanced Python Functions & Strict Type Hinting (Pydantic)

Welcome to Day 1 of your AI Engineering journey!

## Core Theory (Just-in-Time)

**Why Type Hinting and Pydantic?**
In traditional software engineering, inputs are often predictable (e.g., from a controlled frontend or internal API). In AI Engineering, however, your most critical component—the Large Language Model (LLM)—outputs probabilistic, unstructured, and often unpredictable text. 

To build reliable AI pipelines, you must enforce structure at the boundaries. 

**Pydantic** is the industry standard for this. It goes beyond static type checking (like `mypy`); it performs runtime data validation. If an LLM is supposed to return a JSON object containing a `score` (int) and a `summary` (string), Pydantic ensures that output actually conforms to that schema, throwing a clear error if the LLM hallucinated a different format. 

**Advanced Functions (Decorators & *args/**kwargs)**
AI workflows often require cross-cutting concerns like logging execution time (LLM calls are slow!), retrying failed calls (network hiccups or rate limits), or caching. Decorators are the clean, Pythonic way to wrap function execution without cluttering the business logic.

## Code Implementation

Let's walk through a tiered progression of how to apply strict type hinting and decorators in your code, moving from basic concepts to production-ready patterns.

### Basic: Isolating the Core Concept
Here is a simple Pydantic model enforcing type validation with minimal boilerplate.

In [1]:
from pydantic import BaseModel, ValidationError

class SimpleResponse(BaseModel):
    score: int
    summary: str

# Validating correct input
valid_data = {"score": 95, "summary": "Great result"}
parsed_valid = SimpleResponse(**valid_data)
print("Basic Valid:", parsed_valid)

# Validating incorrect input
try:
    invalid_data = {"score": "ninety-five", "summary": "Bad result"}
    SimpleResponse(**invalid_data)
except ValidationError as e:
    print("\nBasic Error:\n", e)

Basic Valid: score=95 summary='Great result'

Basic Error:
 1 validation error for SimpleResponse
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='ninety-five', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing


### Medium: Interacting Concepts
Now, let's combine a simple timing decorator with our Pydantic model to see how decorators and validation interact.

In [2]:
import time
from typing import Callable, Any

def simple_timer(func: Callable[..., Any]) -> Callable[..., Any]:
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start = time.time()
        result = func(*args, **kwargs)
        print(f"[{func.__name__}] took {time.time() - start:.4f}s")
        return result
    return wrapper

@simple_timer
def process_data(data: dict) -> SimpleResponse:
    # Simulate some work
    time.sleep(0.1)
    return SimpleResponse(**data)

processed = process_data({"score": 100, "summary": "Perfect"})
print("Medium Processed:", processed)

[process_data] took 0.1003s
Medium Processed: score=100 summary='Perfect'


### Advanced: Production-Grade Implementation
This example provides a full production-ready implementation, including robust logging, strict type hinting, docstrings, detailed Pydantic `Field` constraints, and error handling.

In [3]:
import time
import logging
from typing import Callable, Any
from functools import wraps
from pydantic import BaseModel, Field, ValidationError

# Setup basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def log_execution_time(func: Callable[..., Any]) -> Callable[..., Any]:
    """
    A decorator that logs the execution time of a function.
    Crucial for monitoring LLM response times in production.
    """
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.time()
        logger.info(f"Executing {func.__name__}...")
        
        try:
            result = func(*args, **kwargs)
        except Exception as e:
            logger.error(f"Error in {func.__name__}: {str(e)}")
            raise e
            
        end_time = time.time()
        logger.info(f"Finished {func.__name__} in {end_time - start_time:.4f} seconds")
        return result
    return wrapper

# --- Pydantic Models for Strict Validation ---

class LLMResponse(BaseModel):
    """
    Defines the exact structure we expect from our LLM.
    Using Field allows us to add constraints and descriptions.
    """
    sentiment: str = Field(..., description="Must be 'positive', 'negative', or 'neutral'")
    confidence_score: float = Field(..., ge=0.0, le=1.0, description="Score between 0.0 and 1.0")
    key_entities: list[str] = Field(default_factory=list, description="List of named entities found in text")

@log_execution_time
def process_llm_output(raw_output: dict[str, Any]) -> LLMResponse:
    """
    Simulates processing a raw dictionary (e.g., parsed JSON from an LLM)
    and validating it against our strict Pydantic schema.
    """
    try:
        # This will automatically validate types, enforce constraints (like ge=0.0),
        # and raise a ValidationError if the data is bad.
        validated_data = LLMResponse(**raw_output)
        return validated_data
    except ValidationError as e:
        logger.error("LLM output failed validation!")
        # In production, you might implement a retry mechanism here to ask the LLM to fix its output.
        raise

# --- Example Usage ---

if __name__ == "__main__":
    # 1. Valid Output
    print("\n--- Testing Valid Output ---")
    good_output = {
        "sentiment": "positive",
        "confidence_score": 0.95,
        "key_entities": ["LangChain", "Pydantic"]
    }
    processed = process_llm_output(good_output)
    print(f"Success! Parsed object: {processed}")

    # 2. Invalid Output (Confidence score too high, missing sentiment)
    print("\n--- Testing Invalid Output ---")
    bad_output = {
        "confidence_score": 1.5, # > 1.0
        "key_entities": ["Error"]
        # Missing sentiment
    }
    
    try:
        process_llm_output(bad_output)
    except ValidationError as e:
        print(f"\nCaught expected validation error:\n{e}")


2026-08-19 12:50:41,785 - INFO - Executing process_llm_output...



--- Testing Valid Output ---


2026-08-19 12:50:41,786 - INFO - Finished process_llm_output in 0.0012 seconds


2026-08-19 12:50:41,787 - INFO - Executing process_llm_output...


2026-08-19 12:50:41,788 - ERROR - LLM output failed validation!


2026-08-19 12:50:41,788 - ERROR - Error in process_llm_output: 2 validation errors for LLMResponse
sentiment
  Field required [type=missing, input_value={'confidence_score': 1.5,...ey_entities': ['Error']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
confidence_score
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


Success! Parsed object: sentiment='positive' confidence_score=0.95 key_entities=['LangChain', 'Pydantic']

--- Testing Invalid Output ---

Caught expected validation error:
2 validation errors for LLMResponse
sentiment
  Field required [type=missing, input_value={'confidence_score': 1.5,...ey_entities': ['Error']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
confidence_score
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


## Common Pitfalls in Production

1.  **Trusting LLM JSON Natively:** Never use raw `json.loads(llm_response)` and just assume the keys you need are present. LLMs hallucinate keys, change types (e.g., sending the string `"true"` instead of the boolean `True`), or return lists instead of dicts. Always pass the raw dict into a Pydantic model immediately.
2.  **Overuse of `Any` or `dict` Type Hints:** Using `def process(data: dict) -> dict:` defeats the purpose of type hinting. If you don't know the exact schema, use Pydantic's `model_validator` or define a flexible schema, but try to avoid raw dicts moving through your core business logic.
3.  **Mutable Defaults in Standard Python:** Remembering that `def func(items=[])` creates a single shared list across all function calls. Pydantic handles this elegantly with `default_factory=list` (as seen in the example above).

## Reference Links

- [Pydantic Official Documentation](https://docs.pydantic.dev/latest/)
- [Python Type Hinting (PEP 484)](https://peps.python.org/pep-0484/)
- [Python Decorators (Real Python)](https://realpython.com/primer-on-python-decorators/)

## Practical Lab / Homework

**Task:** Refactor a legacy document processing script.

Below is a messy, legacy Python script that simulates fetching document metadata and a summary from an "API" (which in the future will be an LLM). 

**Your Requirements:**
1.  **Strict Typing:** Replace the raw dictionary return type with a Pydantic `BaseModel` called `DocumentSummary`.
    *   It should have a `doc_id` (string), `title` (string), `tags` (list of strings), and `word_count` (integer greater than 0).
2.  **Decorator:** Write a retry decorator called `@retry_on_failure` that will retry the function up to 3 times if an Exception is thrown, with a 1-second delay between attempts.
3.  **Refactor:** Apply your Pydantic model and decorator to the `fetch_document_summary` function.

In [4]:
# --- LEGACY CODE (Refactor this!) ---

import random
import time

def fetch_document_summary(doc_id):
    """Simulates a flaky API call that returns unstructured data."""
    # Simulate flakiness
    if random.random() < 0.5:
        raise ConnectionError("API Connection reset!")
        
    # Returns a messy dict
    return {
        "id": doc_id,
        "title": "Understanding Attention Mechanisms",
        "tags": ["AI", "Transformers", "NLP"],
        "words": 1500
    }


# --- YOUR REFACTORED CODE BELOW ---
from pydantic import BaseModel, Field, ValidationError
from typing import Callable, Any
from functools import wraps

class DocumentSummary(BaseModel):
    doc_id: str
    title: str
    tags: list[str] = Field(default_factory=list)
    word_count: int = Field(..., gt=0)

def retry_on_failure(max_retries: int = 3, delay: float = 1.0) -> Callable[..., Any]:
    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            retries = 0
            while retries < max_retries:
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    retries += 1
                    print(f"Attempt {retries} failed: {e}. Retrying in {delay} seconds...")
                    time.sleep(delay)
            raise Exception(f"Failed after {max_retries} attempts")
        return wrapper
    return decorator

@retry_on_failure(max_retries=3, delay=0.5)
def fetch_and_validate_document_summary(doc_id: str) -> DocumentSummary:
    raw_data = fetch_document_summary(doc_id)
    
    # Map the legacy dictionary keys to our Pydantic model
    mapped_data = {
        "doc_id": raw_data.get("id", ""),
        "title": raw_data.get("title", ""),
        "tags": raw_data.get("tags", []),
        "word_count": raw_data.get("words", 0)
    }
    
    return DocumentSummary(**mapped_data)

# --- Test your refactored code ---
if __name__ == "__main__":
    try:
        summary = fetch_and_validate_document_summary("doc-123")
        print(f"\nSuccess! Validated Summary: {summary}")
    except Exception as e:
        print(f"\nFinal Failure: {e}")


Attempt 1 failed: API Connection reset!. Retrying in 0.5 seconds...



Success! Validated Summary: doc_id='doc-123' title='Understanding Attention Mechanisms' tags=['AI', 'Transformers', 'NLP'] word_count=1500
